In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

In [2]:
names = open('data/names.txt', 'r').read().splitlines()
names[:5]

['rosaline', 'jami-pekka', 'meemi', 'sävele', 'mawada']

In [3]:
vocabulary = sorted(list(set(''.join(names))))
vocabulary.insert(0, '<S>')
vocabulary.insert(1, '<E>')
len(vocabulary)

54

In [4]:
atoi = {c: i for i, c in enumerate(vocabulary)}
itoa = {i: c for i, c in enumerate(vocabulary)}

In [5]:
block_size = 8

def build_dataset(words):  
  X, Y = [], []
  
  for w in words:
    context = [0] * 8

    for i in range(len(w) + 1):
      ch = w[i] if i < len(w) else '<E>'
      ix = atoi[ch]
      X.append(context)
      Y.append(ix)
      context = context[1:] + [ix]

  X = torch.tensor(X)
  Y = torch.tensor(Y)
  
  print(X.shape, Y.shape)

  return X, Y

n1 = int(0.8*len(names))
n2 = int(0.9*len(names))

Xtr,  Ytr  = build_dataset(names[:n1])     # 80%
Xdev, Ydev = build_dataset(names[n1:n2])   # 10%
Xte,  Yte  = build_dataset(names[n2:])     # 10%

torch.Size([146520, 8]) torch.Size([146520])
torch.Size([18287, 8]) torch.Size([18287])
torch.Size([18450, 8]) torch.Size([18450])


In [6]:
for x,y in zip(Xtr[:20], Ytr[:20]):
    print(''.join(itoa[int(ix.item())] for ix in x), '-->', itoa[int(y.item())])

<S><S><S><S><S><S><S><S> --> r
<S><S><S><S><S><S><S>r --> o
<S><S><S><S><S><S>ro --> s
<S><S><S><S><S>ros --> a
<S><S><S><S>rosa --> l
<S><S><S>rosal --> i
<S><S>rosali --> n
<S>rosalin --> e
rosaline --> <E>
<S><S><S><S><S><S><S><S> --> j
<S><S><S><S><S><S><S>j --> a
<S><S><S><S><S><S>ja --> m
<S><S><S><S><S>jam --> i
<S><S><S><S>jami --> -
<S><S><S>jami- --> p
<S><S>jami-p --> e
<S>jami-pe --> k
jami-pek --> k
ami-pekk --> a
mi-pekka --> <E>


In [ ]:
import math

class AttentionHead(nn.Module):
    def __init__(self, dim_key, dim_value):
        super().__init__()

        self.dim_key = dim_key
        self.div_value = dim_value

        self.q = nn.Linear(dim_key, dim_key)
        self.k = nn.Linear(dim_key, dim_key)
        self.v = nn.Linear(dim_value, dim_value)
        
    def forward(self, x):
        qk = self.q(x) @ self.k(x).transpose(-2, -1)
        qk = qk / (self.dim_key ** 0.5)
        masked = torch.tril(qk)
        weights = F.softmax(masked, dim=-1)
        v = self.v(x)
        out = weights @ v
        
        return out

class PositionalEncoding(nn.Module):

    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 5000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        position = torch.arange(max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe = torch.zeros(max_len, 1, d_model)
        pe[:, 0, 0::2] = torch.sin(position * div_term)
        pe[:, 0, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Arguments:
            x: Tensor, shape ``[seq_len, batch_size, embedding_dim]``
        """
        x = x + self.pe[:x.size(0)]
        return self.dropout(x)

class Transformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, nheads):
        super().__init__()

        self.embed_dim = embed_dim

        self.positional_encoding = PositionalEncoding(embed_dim)
        self.token_embedding = nn.Embedding(vocab_size, embed_dim)
        self.heads = nn.ModuleList([AttentionHead(embed_dim, embed_dim) for _ in range(nheads)])
        self.wo = nn.Linear(embed_dim * nheads, embed_dim)
        self.logits = nn.Linear(embed_dim, vocab_size)
        self.norm = nn.LayerNorm(embed_dim)
        self.fc = nn.Linear(embed_dim, embed_dim)
        
    def forward(self, x):
        x = self.token_embedding(x) * math.sqrt(self.embed_dim)
        x = self.positional_encoding(x)

        head_outputs = []
        for head in self.heads:
            head_outputs.append(head(x))

        concat = torch.stack(head_outputs)
        qkv = self.wo(concat)

        norm = self.norm(qkv)
        x = self.fc(norm)
        logits = self.logits(x)

        return logits

In [ ]:
CONTEXT     = 8
EMBED_DIM   = 32
NHEADS      = 4
BATCH_SIZE  = 64
EPOCHS      = 10_000
LR          = 1e-4

vocab_size = len(vocabulary)

print(f"Vocabulary size: {vocab_size}")
print(f"Training samples: {len(Xtr)}")

model = Transformer(vocab_size, embed_dim=EMBED_DIM, nheads=NHEADS)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

num_params = sum(p.numel() for p in model.parameters())
print(f"Parameters: {num_params:,}")

losses = []
val_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()

    idx  = torch.randint(0, len(Xtr), (BATCH_SIZE,))
    xb, yb = Xtr[idx], Ytr[idx]

    logits = model(xb)
    logits = logits[:, :, -1]

    loss = F.cross_entropy(logits, yb)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

    if epoch % 500 == 0:
        with torch.no_grad():
            model.eval()

            dev_logits = model(Xdev)
            dev_logits = dev_logits[:, :, -1]
            dev_loss = F.cross_entropy(dev_logits, Ydev)
            val_losses.append(dev_loss.item())

        print(f"Epoch {epoch:4d} | loss {loss.item():.4f} | dev loss {dev_loss.item():.4f}")

Vocabulary size: 54
Training samples: 146520
Parameters: 325,878
Epoch  500 | loss 2.2091 | dev loss 2.4655
Epoch 1000 | loss 2.1465 | dev loss 2.3680
Epoch 1500 | loss 2.3925 | dev loss 2.3045
Epoch 2000 | loss 2.3368 | dev loss 2.2698
Epoch 2500 | loss 2.4711 | dev loss 2.2350
Epoch 3000 | loss 2.1295 | dev loss 2.2125
Epoch 3500 | loss 2.4186 | dev loss 2.1897
Epoch 4000 | loss 1.9231 | dev loss 2.1705
Epoch 4500 | loss 2.2264 | dev loss 2.1624
Epoch 5000 | loss 2.0177 | dev loss 2.1476
Epoch 5500 | loss 2.0110 | dev loss 2.1252
Epoch 6000 | loss 1.9142 | dev loss 2.1172
Epoch 6500 | loss 2.1678 | dev loss 2.1043
Epoch 7000 | loss 2.1806 | dev loss 2.0985
Epoch 7500 | loss 1.7657 | dev loss 2.0872
Epoch 8000 | loss 1.7952 | dev loss 2.0774
Epoch 8500 | loss 2.2962 | dev loss 2.0678
Epoch 9000 | loss 2.2192 | dev loss 2.0633
Epoch 9500 | loss 2.1385 | dev loss 2.0562
Epoch 10000 | loss 1.7206 | dev loss 2.0488
